In [3]:
import pandas as pd
df = pd.read_csv('train.csv')
df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 8693 entries, 0 to 8692
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   PassengerId   8693 non-null   str    
 1   HomePlanet    8492 non-null   str    
 2   CryoSleep     8476 non-null   object 
 3   Cabin         8494 non-null   str    
 4   Destination   8511 non-null   str    
 5   Age           8514 non-null   float64
 6   VIP           8490 non-null   object 
 7   RoomService   8512 non-null   float64
 8   FoodCourt     8510 non-null   float64
 9   ShoppingMall  8485 non-null   float64
 10  Spa           8510 non-null   float64
 11  VRDeck        8505 non-null   float64
 12  Name          8493 non-null   str    
 13  Transported   8693 non-null   bool   
dtypes: bool(1), float64(6), object(2), str(5)
memory usage: 891.5+ KB


,PassengerId,HomePlanet,CryoSleep,Cabin,Destination,Age,VIP,RoomService,FoodCourt,ShoppingMall,Spa,VRDeck,Name,Transported
0,0001_01,Europa,False,B/0/P,TRAPPIST-1e,39.0,False,0.0,0.0,0.0,0.0,0.0,Maham Ofracculy,False
1,0002_01,Earth,False,F/0/S,TRAPPIST-1e,24.0,False,109.0,9.0,25.0,549.0,44.0,Juanna Vines,True
2,0003_01,Europa,False,A/0/S,TRAPPIST-1e,58.0,True,43.0,3576.0,0.0,6715.0,49.0,Altark Susent,False
3,0003_02,Europa,False,A/0/S,TRAPPIST-1e,33.0,False,0.0,1283.0,371.0,3329.0,193.0,Solam Susent,False
4,0004_01,Earth,False,F/1/S,TRAPPIST-1e,16.0,False,303.0,70.0,151.0,565.0,2.0,Willy Santantines,True


In [5]:
y = df['Transported'].astype(int)
x = df.drop(columns=['PassengerId','Name','Transported'])
def extract_cabin_features(cabin):
    if pd.isna(cabin):
        return pd.Series(['Unknown',-1,'Unknown'])
    parts = cabin.split('/')
    if len(parts) == 3:
        deck, room,side = parts
        return pd.Series([deck,int(room),side])
    else:
        return pd.Series(['Unknown',-1,'Unknown'])

cabin_features = x['Cabin'].apply(extract_cabin_features)
cabin_features.columns = ['Deck','RoomNumber','Side']
x = pd.concat([x.drop(columns=['Cabin']),cabin_features],axis=1)

x['CryoSleep'] = x['CryoSleep'].map({True:1,False:0})
x['VIP'] = x['VIP'].map({True:1,False:0})

expense_cols = ['RoomService','FoodCourt','ShoppingMall','Spa','VRDeck']
x[expense_cols] = x[expense_cols].fillna(0)
x['TotalSpending'] = x[expense_cols].sum(axis=1)

In [7]:
x['Age'] = x['Age'].fillna(x['Age'].median())
x['HomePlanet'] = x['HomePlanet'].fillna(x['HomePlanet'].mode()[0])
x['Destination'] = x['Destination'].fillna(x['Destination'].mode()[0])
x['RoomNumber'] = x['RoomNumber'].astype(int)

In [8]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

categorical_cols = ['HomePlanet','Destination','Deck','Side']
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    x[col] = x[col].fillna('Unknown')
    x[col] = le.fit_transform(x[col])
    label_encoders[col] = le

numeric_cols = ['Age','RoomNumber','RoomService','FoodCourt','Spa','ShoppingMall','VRDeck','TotalSpending']
scaler = StandardScaler()
x[numeric_cols] = scaler.fit_transform(x[numeric_cols])

In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)
rf = RandomForestClassifier(n_estimators=100,max_depth=15,random_state=42)
rf.fit(x_train,y_train)

y_pred = rf.predict(x_test)
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)
print(f'Accuracy: {accuracy_score(y_test,y_pred)*100:.2f}%')
importance = pd.Series(rf.feature_importances_,index=x_train.columns)
print('\nTop 20 Feature importance:')
print(importance.sort_values(ascending=False).head(10))

Accuracy: 79.18%

Top 20 Feature importance:
TotalSpending    0.144279
RoomNumber       0.131129
Age              0.099362
CryoSleep        0.094675
Spa              0.087658
FoodCourt        0.083351
VRDeck           0.082144
RoomService      0.073804
ShoppingMall     0.067228
Deck             0.055786
dtype: float64


In [11]:
test_df = pd.read_csv('test.csv')
test_ids = test_df['PassengerId']
x_test = test_df.drop(columns=['PassengerId', 'Name'])

# ----------------------
# 对测试集做完全一样的预处理！
# ----------------------
# 舱位处理
cabin_features_test = x_test['Cabin'].apply(extract_cabin_features)
cabin_features_test.columns = ['Deck', 'RoomNumber', 'Side']
x_test = pd.concat([x_test.drop(columns=['Cabin']), cabin_features_test], axis=1)

# 布尔值转换
x_test['CryoSleep'] = x_test['CryoSleep'].map({True: 1, False: 0})
x_test['VIP'] = x_test['VIP'].map({True: 1, False: 0})

# 消费特征
x_test[expense_cols] = x_test[expense_cols].fillna(0)
x_test['TotalSpending'] = x_test[expense_cols].sum(axis=1)

# 缺失值
x_test['Age'] = x_test['Age'].fillna(df['Age'].median())
x_test['HomePlanet'] = x_test['HomePlanet'].fillna(df['HomePlanet'].mode()[0])
x_test['Destination'] = x_test['Destination'].fillna(df['Destination'].mode()[0])
x_test['RoomNumber'] = x_test['RoomNumber'].astype(int)

# 类别编码（使用训练集的编码器！）
for col in categorical_cols:
    le = label_encoders[col]
    x_test[col] = x_test[col].fillna('Unknown')
    x_test[col] = le.transform(x_test[col])

# 标准化（使用训练集的 scaler）
x_test[numeric_cols] = scaler.transform(x_test[numeric_cols])

# 确保列顺序和训练集一致
x_test = x_test[x_train.columns]

# ======================
# 10. 预测并生成提交文件
# ======================
test_pred = rf.predict(x_test)
predicted = test_pred.astype(bool)

submission = pd.DataFrame({
    'PassengerId': test_ids,
    'Transported': predicted
})

submission.to_csv('submission.csv', index=False)
print("\n✅ 预测完成！文件 submission.csv 已生成！")


✅ 预测完成！文件 submission.csv 已生成！
